## This is the very first step to download data


In [0]:
# Use Volumes instead of DBFS root
spark.sql("CREATE CATALOG IF NOT EXISTS movie_recsys")
spark.sql("CREATE SCHEMA IF NOT EXISTS movie_recsys.data")
spark.sql("CREATE VOLUME IF NOT EXISTS movie_recsys.data.raw")
spark.sql("CREATE VOLUME IF NOT EXISTS movie_recsys.data.artifacts")
spark.sql("CREATE VOLUME IF NOT EXISTS movie_recsys.data.outputs")

print("Volumes created. Path to use: /Volumes/movie_recsys/data/raw/")

In [0]:
#Cell 2 — download the reviews file

import urllib.request
import os

url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Movies_and_TV.jsonl.gz"
dest = "/Volumes/movie_recsys/data/raw/Movies_and_TV.jsonl.gz"

print("Downloading... this takes 10-20 minutes.")
urllib.request.urlretrieve(url, dest)

size_gb = os.path.getsize(dest) / (1024**3)
print(f"Done. File size: {size_gb:.2f} GB")

In [0]:
import urllib.request
import os

url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Movies_and_TV.jsonl.gz"
dest = "/Volumes/movie_recsys/data/raw/meta_Movies_and_TV.jsonl.gz"

print("Downloading metadata... this takes 5-10 minutes.")
urllib.request.urlretrieve(url, dest)

size_gb = os.path.getsize(dest) / (1024**3)
print(f"Done. File size: {size_gb:.2f} GB")

In [0]:
# Cell 4 — validate both files are readable and check the price field:

import gzip
import json

# Check reviews
reviews_path = "/Volumes/movie_recsys/data/raw/Movies_and_TV.jsonl.gz"
meta_path = "/Volumes/movie_recsys/data/raw/meta_Movies_and_TV.jsonl.gz"

# Read first 3 records from each
print("=== REVIEWS - first record keys and sample ===")
with gzip.open(reviews_path, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        rec = json.loads(line.strip())
        if i == 0:
            print(f"Keys: {list(rec.keys())}")
            for k, v in rec.items():
                print(f"  {k}: {str(v)[:80]}")
        if i >= 2:
            break

print("\n=== METADATA - first record keys and sample ===")
with gzip.open(meta_path, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        rec = json.loads(line.strip())
        if i == 0:
            print(f"Keys: {list(rec.keys())}")
            for k, v in rec.items():
                print(f"  {k}: {str(v)[:80]}")
        if i >= 2:
            break

In [0]:
# Cell 5 — quick count of both files. This takes 5-10 minutes:

import gzip
import json

def count_records(path, name):
    count = 0
    price_non_null = 0
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line.strip())
            count += 1
            if name == "metadata":
                p = rec.get('price')
                if p and str(p) not in ('None', ''):
                    price_non_null += 1
            if count % 1_000_000 == 0:
                print(f"  {name}: {count:,} records so far...")
    return count, price_non_null

review_count, _ = count_records(reviews_path, "reviews")
meta_count, price_count = count_records(meta_path, "metadata")

print(f"\n{'='*40}")
print(f"Reviews:  {review_count:,} records")
print(f"Metadata: {meta_count:,} records")
print(f"Metadata with price: {price_count:,} ({100*price_count/meta_count:.1f}%)")
print(f"{'='*40}")



In [0]:

# save the validation report and close the notebook
import json
from datetime import datetime

report = {
    "timestamp": datetime.now().isoformat(),
    "review_count": 17_328_314,
    "meta_count": 748_224,
    "meta_with_price_count": 263_941,
    "meta_price_coverage_pct": 35.3,
    "reviews_path": "/Volumes/movie_recsys/data/raw/Movies_and_TV.jsonl.gz",
    "meta_path": "/Volumes/movie_recsys/data/raw/meta_Movies_and_TV.jsonl.gz",
    "status": "validated",
    "note_reviews": "17.3M includes Prime Video streaming content",
    "note_price": "35.3% price coverage. Revenue tier model uses non-null subset.",
    "next_notebook": "01_eda_and_filtering"
}

out = "/Volumes/movie_recsys/data/outputs/00_validation_report.json"
with open(out, 'w') as f:
    json.dump(report, f, indent=2)

print("Validation complete. Ready for next notebook.")
print(json.dumps(report, indent=2))